# Network Analysis - Cathy

> COMP 4710 Group 11 - Progress Report 1

## Setup

In [ ]:
# Parameters
save_figures = False
figures_dir = "reports/pr1/figures"
dpi = 200

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx

repo_root = Path.cwd()
if not (repo_root / "ptn_analysis").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

figure_output_directory = Path(figures_dir)
if not figure_output_directory.is_absolute():
    figure_output_directory = (repo_root / figure_output_directory).resolve()
figure_output_directory.mkdir(parents=True, exist_ok=True)

In [ ]:
import contextily as cx
import geopandas as gpd
from shapely.geometry import Point

from ptn_analysis import TransitContext
from ptn_analysis.context.serving import MapDataLoader
from ptn_analysis.analysis.visualization import save_report_figure, add_consistent_basemap

city_key = "ywg"
feed_id = "current"
ctx = TransitContext.from_defaults(feed_id=feed_id)
network_analyzer = ctx.network()

In [ ]:
def save_if_requested(figure, filename: str) -> None:
    """Save a figure only when papermill export is enabled.

    Args:
        figure: Matplotlib figure to save.
        filename: Output file name inside the report figure directory.

    Returns:
        None.
    """
    if save_figures:
        save_report_figure(figure, str(figure_output_directory / filename), dpi=dpi)

## Build Network Graph

In [ ]:
stats = network_analyzer.stats()
metrics_table = network_analyzer.build_network_metrics_table()
comparison_table = network_analyzer.build_network_comparison_table(baseline_feed_id="avg_pre_ptn")
print(f"Graph: {stats['node_count']:,} nodes, {stats['edge_count']:,} edges")
print(f"Density: {stats['density']:.4f}")
print(f"Average degree: {stats['avg_degree']:.2f}")
display(metrics_table)
display(comparison_table)

## Centrality Analysis

In [ ]:
degree_table = network_analyzer.degree_centrality()
stop_table = network_analyzer.stops_df()
degree_table = degree_table.merge(stop_table[["stop_id", "stop_name"]], on="stop_id", how="left")
betweenness_table = network_analyzer.betweenness_centrality()
communities_table = network_analyzer.detect_communities()
hub_table = network_analyzer.build_top_hub_table(n=20)
display(hub_table.head(10))
display(betweenness_table.head(10))

## Report Figures

In [ ]:
connection_map = MapDataLoader(city_key=city_key, feed_id=feed_id, db_instance=ctx.working_db).load_connections()
plot_connections = connection_map.sort_values("stop_connection_count", ascending=False).head(400)

# Convert hub locations to Web Mercator for basemap overlay
hub_points = gpd.GeoDataFrame(
    hub_table,
    geometry=[Point(lon, lat) for lon, lat in zip(hub_table["stop_lon"], hub_table["stop_lat"])],
    crs="EPSG:4326",
).to_crs(epsg=3857)

figure, axis = plt.subplots(figsize=(10.5, 10.5))

# Draw connection lines in Web Mercator
for row in plot_connections.itertuples(index=False):
    from_pt = gpd.GeoDataFrame(
        [{"geometry": Point(row.from_lon, row.from_lat)}], crs="EPSG:4326"
    ).to_crs(epsg=3857).geometry.iloc[0]
    to_pt = gpd.GeoDataFrame(
        [{"geometry": Point(row.to_lon, row.to_lat)}], crs="EPSG:4326"
    ).to_crs(epsg=3857).geometry.iloc[0]
    line_width = max(0.2, min(2.0, row.stop_connection_count / 200.0))
    axis.plot([from_pt.x, to_pt.x], [from_pt.y, to_pt.y], color="#bdbdbd", alpha=0.18, linewidth=line_width)

axis.scatter(
    hub_points.geometry.x, hub_points.geometry.y,
    s=hub_table["total_degree"] * 4, c="#0064B1", alpha=0.8,
    edgecolor="white", linewidth=0.4,
)

# Zoom to data bounds
bounds = hub_points.total_bounds
x_pad = (bounds[2] - bounds[0]) * 0.12
y_pad = (bounds[3] - bounds[1]) * 0.12
axis.set_xlim(bounds[0] - x_pad, bounds[2] + x_pad)
axis.set_ylim(bounds[1] - y_pad, bounds[3] + y_pad)

add_consistent_basemap(axis, zoom=12)
axis.set_title("Current Network Connections and Hub Stops")
save_if_requested(figure, "network_map.png")
figure

In [ ]:
hub_stop_ids = hub_table["stop_id"].tolist()
hub_graph = network_analyzer.graph.subgraph(hub_stop_ids).copy()
community_lookup = dict(zip(communities_table["stop_id"], communities_table["community_id"]))
degree_lookup = dict(zip(hub_table["stop_id"], hub_table["total_degree"]))
name_lookup = dict(zip(hub_table["stop_id"], hub_table["stop_name"]))
palette = ["#0064B1", "#F37043", "#00B262", "#026C7E", "#6450A1", "#F27EA6", "#8B6914"]

figure, axis = plt.subplots(figsize=(10, 8))
if hub_graph.number_of_edges() == 0:
    axis.barh(hub_table["stop_name"], hub_table["total_degree"], color="#0064B1")
    axis.set_title("Top Hub Stops by Total Degree")
    axis.set_xlabel("Total degree")
else:
    position = nx.spring_layout(hub_graph, seed=42, k=1.1)
    node_colors = []
    node_sizes = []
    for node_id in hub_graph.nodes:
        community_id = community_lookup.get(node_id, 0)
        node_colors.append(palette[community_id % len(palette)])
        node_sizes.append(max(80, degree_lookup.get(node_id, 1) * 7))
    nx.draw_networkx_edges(hub_graph, position, ax=axis, alpha=0.25, edge_color="#9e9e9e")
    nx.draw_networkx_nodes(hub_graph, position, ax=axis, node_color=node_colors, node_size=node_sizes, edgecolors="white", linewidths=0.5)
    label_lookup = {}
    for node_id in hub_table.head(10)["stop_id"].tolist():
        label_lookup[node_id] = name_lookup.get(node_id, str(node_id))
    nx.draw_networkx_labels(hub_graph, position, labels=label_lookup, font_size=7, ax=axis)
    axis.set_title("Hub Subgraph with Louvain Communities")
axis.set_axis_off()
save_if_requested(figure, "network_graph.png")
figure

This notebook now depends only on `NetworkAnalyzer` and the shared map helpers. The old notebook-local graph setup has been removed.